# 🍅 Analizador de Calidad de Frutas: Modo Foto (Estático)

Este cuaderno implementa un sistema de visión artificial para analizar imágenes estáticas (fotos tomadas con celular o cámara) de jitomates.

**Diferencia con el modo Webcam:**
* **Alta Resolución:** Aquí procesamos imágenes grandes, permitiendo detectar detalles finos como defectos en la piel.
* **Interacción:** Usamos ventanas de diálogo (`tkinter`) para seleccionar archivos desde tu computadora.
* **Análisis Profundo:** Aplicamos reglas más estrictas de forma y color.

## 1. Importación de Librerías

Además de **OpenCV** y **NumPy**, ahora importamos **Tkinter**. 
* `tkinter`: Es la biblioteca estándar de interfaz gráfica de Python. La usaremos únicamente para abrir la ventanita de "Abrir Archivo".

In [ ]:
import cv2
import numpy as np
import tkinter as tk
from tkinter import filedialog
import os

print("Librerías cargadas. Listo para iniciar.")

## 2. Configuración Maestra (CFG)

Este diccionario controla todo el comportamiento del algoritmo. 

### Puntos Clave Explicados:
1.  **`TOMATO_MIN_SOLIDITY` (0.88):** Un jitomate es un objeto sólido y convexo. Si el objeto tiene formas extrañas (como una mano abierta), su solidez bajará. Exigimos 88% de solidez.
2.  **`USE_CALYX_CHECK`:** Activa la búsqueda del tallo/cáliz. Es crucial para distinguir un jitomate rojo de una manzana roja.
3.  **`DEFECT_IGNORE_TOP_FRAC` (0.22):** Al buscar defectos (manchas oscuras), ignoramos el 22% superior de la fruta. **¿Por qué?** Porque ahí está el tallo, que es oscuro por naturaleza. Si no lo ignoramos, el sistema pensará que el tallo es un hongo o un golpe.

In [ ]:
# ============================================================
# CONFIGURACIÓN GENERAL (AJUSTABLE)
# ============================================================

CFG = {
    # ---------- Visualización ----------
    "TARGET_VIEW_W": 1200,        # Redimensionar imagen para que quepa en pantalla
    "MAX_VIEW_W": 1600,
    "MAX_VIEW_H": 900,
    "ALLOW_UPSCALE": True,
    "PANEL_H": 95,                # Altura del panel de información superior
    "ALPHA_PANEL": 0.80,          # Transparencia del panel
    "FONT": cv2.FONT_HERSHEY_SIMPLEX,

    # ---------- Segmentación (Separar objeto del fondo) ----------
    "KERNEL": 5,           # Tamaño del filtro de limpieza
    "OPEN_IT": 2,          # Eliminar ruido blanco externo
    "CLOSE_IT": 3,         # Cerrar huecos negros internos
    "MIN_AREA": 2500,      # Ignorar objetos muy pequeños
    "MIN_EXTENT": 0.35,    # Relación área/rectángulo
    "BORDER_FRAC": 0.06,   # Qué tanto del borde usamos para detectar el fondo

    # ---------- Forma: Geometría del Jitomate ----------
    "TOMATO_MIN_SOLIDITY": 0.88, 
    "TOMATO_MIN_CIRC": 0.42,
    "TOMATO_MAX_ASPECT": 1.65,    # No debe ser muy alargado (pepino)
    "TOMATO_MIN_EXTENT": 0.55,
    "TOMATO_SHAPE_SCORE_MIN": 0.55, # Puntuación mínima de forma

    # ---------- El Cáliz (La estrella verde) ----------
    "USE_CALYX_CHECK": True,
    "CALYX_TOP_FRAC": 0.30,       # Solo buscar en el 30% superior
    "CALYX_H_MIN": 35, "CALYX_H_MAX": 95, # Rango verde HSV
    "CALYX_MIN_S": 50, "CALYX_MIN_V": 40,
    "CALYX_RATIO_MIN": 0.015,     # Mínimo 1.5% de verde en la zona superior
    "CALYX_WEIGHT": 0.55,         # El cáliz vale el 55% de la decisión final

    # Score final para decir "Es Jitomate"
    "TOMATO_SCORE_MIN": 0.62,

    # ---------- Color (Madurez) ----------
    "MIN_S": 35, "MIN_V": 35,     # Ignorar grises/negros
    "H_RED1_MAX": 10, "H_RED2_MIN": 165,
    "H_ORANGE_MIN": 10, "H_ORANGE_MAX": 25,
    "H_YELLOW_MIN": 25, "H_YELLOW_MAX": 40,
    "H_GREEN_MIN": 35, "H_GREEN_MAX": 95,

    # ---------- Daño (Defectos) ----------
    "DEFECT_IGNORE_TOP_FRAC": 0.22, # Ignorar zona del tallo para no marcarlo como daño
    "DEFECT_RATIO_MIN": 0.030,      # Si >3% es oscuro, está dañado
    "DEFECT_DARK_Q": 0.12,          # Umbral de oscuridad relativo

    # ---------- Colores UI ----------
    "C_RED": (0, 0, 255), "C_ORANGE": (0, 165, 255),
    "C_GREEN": (0, 255, 0), "C_WHITE": (255, 255, 255),
    "C_BLACK": (0, 0, 0), "C_YELLOW": (0, 255, 255),
}

## 3. Utilidades de Sistema

Aquí definimos funciones auxiliares:
1.  `seleccionar_imagen`: Abre el explorador de archivos de Windows/Mac/Linux.
2.  `resize_for_view`: Ajusta fotos enormes (ej. 12 megapíxeles) para que quepan en la pantalla de tu laptop sin deformarse.

In [ ]:
def seleccionar_imagen():
    """Abre una ventana nativa del sistema operativo para elegir archivo."""
    root = tk.Tk()
    root.withdraw() # Ocultar la ventana principal de Tkinter
    path = filedialog.askopenfilename(
        title="Selecciona una imagen de fruta",
        filetypes=[("Imágenes", "*.jpg *.jpeg *.png *.bmp *.tif *.tiff")]
    )
    root.destroy()
    return path

def resize_for_view(img):
    """Estandariza la vista manteniendo la proporción correcta."""
    h, w = img.shape[:2]
    target_w = CFG["TARGET_VIEW_W"]
    scale = target_w / float(w)

    if not CFG["ALLOW_UPSCALE"] and scale > 1.0:
        scale = 1.0

    new_w = int(w * scale)
    new_h = int(h * scale)

    # Ajuste adicional si la altura excede la pantalla
    max_w, max_h = CFG["MAX_VIEW_W"], CFG["MAX_VIEW_H"]
    fit_scale = min(max_w / float(new_w), max_h / float(new_h), 1.0)
    new_w = int(new_w * fit_scale)
    new_h = int(new_h * fit_scale)

    if new_w != w or new_h != h:
        interp = cv2.INTER_CUBIC if scale > 1.0 else cv2.INTER_AREA
        img = cv2.resize(img, (new_w, new_h), interpolation=interp)

    return img

def wait_until_close(win_name):
    """Bucle para mantener la ventana abierta hasta presionar tecla o X."""
    while True:
        # Si la ventana se cerró con la X, salir
        if cv2.getWindowProperty(win_name, cv2.WND_PROP_VISIBLE) < 1:
            return None
        k = cv2.waitKey(30)
        if k != -1: # Si se presionó una tecla
            return k

def put_label(img, text, x, y, fg, bg, scale=0.65, thick=2, pad=6):
    """Dibuja texto con un fondo de color para que sea legible."""
    font = CFG["FONT"]
    (tw, th), _ = cv2.getTextSize(text, font, scale, thick)
    x2 = x + tw + pad * 2
    y2 = y + th + pad * 2
    cv2.rectangle(img, (x, y), (x2, y2), bg, -1)
    cv2.putText(img, text, (x + pad, y + th + pad - 2),
                font, scale, fg, thick, cv2.LINE_AA)

## 4. Análisis Geométrico

Extraemos las métricas matemáticas del contorno.

* **Circularity:** Relación entre área y perímetro. (1.0 = Círculo Perfecto).
* **Solidity:** Relación entre el área del objeto y su "envolvente convexa". Un objeto rugoso o con huecos tiene baja solidez.
* **Aspect Ratio:** Ancho vs Alto. Un jitomate suele ser 1:1 o 1.2:1.

In [ ]:
def contour_metrics(cnt):
    area = cv2.contourArea(cnt)
    if area <= 0: return None

    peri = cv2.arcLength(cnt, True)
    if peri <= 0: return None

    x, y, w, h = cv2.boundingRect(cnt)
    bbox_area = max(1, w * h)

    # Matemáticas de forma
    circ = (4.0 * np.pi * area) / (peri * peri)
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0.0
    extent = area / bbox_area
    aspect = max(w, h) / max(1, min(w, h))

    return {"area": area, "x": x, "y": y, "w": w, "h": h,
            "circ": circ, "solidity": solidity, "extent": extent, "aspect": aspect}

def mask_from_contour(shape, cnt):
    """Crea una imagen negra con el objeto relleno en blanco."""
    m = np.zeros(shape[:2], dtype=np.uint8)
    cv2.drawContours(m, [cnt], -1, 255, -1)
    return m

## 5. Segmentación Robusta (LAB + HSV)

Esta es la función más avanzada para separar la fruta del fondo.

1.  **Muestreo de Borde:** Asumimos que el marco de la imagen (los bordes) son fondo.
2.  **Espacio Lab:** Usamos el espacio de color Lab porque separa la luminosidad (L) del color (a, b). Esto nos permite detectar el color de la fruta incluso si hay sombras o brillos.
3.  **Distancia de Color:** Calculamos qué tan diferente es cada píxel central respecto a los bordes. Si es muy diferente, es fruta.

In [ ]:
def build_object_mask(img_bgr):
    blur = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    # Convertimos a LAB (Luminosidad, a=Verde-Rojo, b=Azul-Amarillo)
    lab = cv2.cvtColor(blur, cv2.COLOR_BGR2LAB)
    hsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)

    A = lab[:, :, 1]
    B = lab[:, :, 2]
    S = hsv[:, :, 1]

    # Definimos el margen del borde
    h, w = img_bgr.shape[:2]
    m = max(5, int(min(h, w) * CFG["BORDER_FRAC"]))

    # Creamos máscara del borde
    border = np.zeros((h, w), dtype=np.uint8)
    border[:m, :] = 1; border[-m:, :] = 1
    border[:, :m] = 1; border[:, -m:] = 1

    # Calculamos el color promedio del fondo
    bgA = float(np.mean(A[border == 1]))
    bgB = float(np.mean(B[border == 1]))

    # Distancia Euclidiana de cada pixel al color del fondo
    dist_ab = np.sqrt((A.astype(np.float32) - bgA) ** 2 + (B.astype(np.float32) - bgB) ** 2)
    dist_u8 = cv2.normalize(dist_ab, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # Binarización automática (Otsu)
    _, mask_ab = cv2.threshold(dist_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, mask_s = cv2.threshold(S, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Combinamos: Diferente al fondo O alta saturación
    mask = cv2.bitwise_or(mask_ab, mask_s)

    # Limpieza morfológica
    k = CFG["KERNEL"]
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=CFG["CLOSE_IT"])
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=CFG["OPEN_IT"])

    # Eliminar ruido pequeño
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    for i in range(1, num):
        if stats[i, cv2.CC_STAT_AREA] < int(CFG["MIN_AREA"] * 0.35):
            mask[labels == i] = 0

    return mask

def extract_instances(mask):
    """Encuentra los objetos individuales en la máscara."""
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    good = []
    for c in cnts:
        m = contour_metrics(c)
        if m is None: continue
        # Filtros de tamaño y forma básica
        if m["area"] < CFG["MIN_AREA"]: continue
        if m["extent"] < CFG["MIN_EXTENT"]: continue
        good.append(c)
    
    # Ordenar por tamaño (el más grande primero)
    good.sort(key=lambda c: cv2.contourArea(c), reverse=True)
    return good

## 6. Lógica Biológica: Detección del Cáliz

Aquí diferenciamos jitomates de manzanas.
* Analizamos solo el **30% superior** del bounding box (`CALYX_TOP_FRAC`).
* Buscamos píxeles **verdes**.
* Si hay suficiente verde arriba, el puntaje de "Probabilidad de ser Jitomate" sube drásticamente.

In [ ]:
def tomato_shape_score(metrics):
    """Puntaje basado puramente en geometría (redondez, solidez)."""
    circ, sol, asp, ext = metrics["circ"], metrics["solidity"], metrics["aspect"], metrics["extent"]

    s_circ = np.clip((circ - CFG["TOMATO_MIN_CIRC"]) / (1.0 - CFG["TOMATO_MIN_CIRC"]), 0, 1)
    s_sol  = np.clip((sol - CFG["TOMATO_MIN_SOLIDITY"]) / (1.0 - CFG["TOMATO_MIN_SOLIDITY"]), 0, 1)
    s_asp  = np.clip((CFG["TOMATO_MAX_ASPECT"] - asp) / (CFG["TOMATO_MAX_ASPECT"] - 1.0), 0, 1)
    s_ext  = np.clip((ext - CFG["TOMATO_MIN_EXTENT"]) / (1.0 - CFG["TOMATO_MIN_EXTENT"]), 0, 1)

    return float(0.30 * s_circ + 0.30 * s_sol + 0.20 * s_asp + 0.20 * s_ext)

def calyx_green_ratio(img_bgr, fruit_mask, cnt):
    """Calcula el porcentaje de verde en la parte superior del fruto."""
    met = contour_metrics(cnt)
    x, y, w, h = met["x"], met["y"], met["w"], met["h"]

    top_h = int(h * CFG["CALYX_TOP_FRAC"])
    if top_h < 10: return 0.0

    # ROI: Recorte de la parte superior
    roi = img_bgr[y:y+top_h, x:x+w]
    roi_mask = fruit_mask[y:y+top_h, x:x+w]

    if roi.size == 0: return 0.0

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    H, S, V = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

    inside = (roi_mask > 0)
    total = int(np.count_nonzero(inside))
    if total < 120: return 0.0

    # Definición de verde para el cáliz
    green = inside & \
            (H >= CFG["CALYX_H_MIN"]) & (H <= CFG["CALYX_H_MAX"]) & \
            (S >= CFG["CALYX_MIN_S"]) & (V >= CFG["CALYX_MIN_V"])

    ratio = float(np.count_nonzero(green)) / float(total)
    return ratio

def tomato_score_total(shape_score, calyx_ratio):
    """Combina la forma y el cáliz para dar un veredicto final."""
    if not CFG["USE_CALYX_CHECK"]:
        return float(shape_score)

    # Si tiene cáliz, el score sube. Si no, baja.
    calyx_score = np.clip(calyx_ratio / max(CFG["CALYX_RATIO_MIN"], 1e-6), 0, 1)
    w = CFG["CALYX_WEIGHT"]
    return float((1 - w) * shape_score + w * calyx_score)

## 7. Madurez y Detección de Defectos

* **Madurez:** Conteo de píxeles rojos vs verdes.
* **Defectos:** Buscamos zonas muy oscuras (V bajo en HSV). 
    * *Truco:* Ignoramos la parte superior (`DEFECT_IGNORE_TOP_FRAC`) para que el tallo natural no cuente como fruta podrida.

In [ ]:
def classify_ripeness(img_bgr, fruit_mask):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(hsv)

    valid = (fruit_mask > 0) & (S > CFG["MIN_S"]) & (V > CFG["MIN_V"])
    total = int(np.count_nonzero(valid))
    if total < 150:
        return "DESCONOCIDO", CFG["C_YELLOW"], 0.0

    h_vals = H[valid]

    # Contamos porcentajes de color
    p_red = np.mean((h_vals <= CFG["H_RED1_MAX"]) | (h_vals >= CFG["H_RED2_MIN"]))
    p_or  = np.mean((h_vals > CFG["H_ORANGE_MIN"]) & (h_vals <= CFG["H_ORANGE_MAX"]))
    p_ye  = np.mean((h_vals > CFG["H_YELLOW_MIN"]) & (h_vals <= CFG["H_YELLOW_MAX"]))
    p_gr  = np.mean((h_vals >= CFG["H_GREEN_MIN"]) & (h_vals <= CFG["H_GREEN_MAX"]))
    p_trans = p_or + p_ye

    # Reglas de decisión
    if p_red >= 0.28 and p_red >= p_trans and p_red >= p_gr:
        return "MADURO", CFG["C_RED"], float(p_red)
    if p_gr >= 0.35 and p_gr >= p_red and p_gr >= p_trans:
        return "INMADURO", CFG["C_GREEN"], float(p_gr)
    if p_trans >= 0.22 and p_trans >= p_red and p_trans >= p_gr:
        return "TRANSICION", CFG["C_ORANGE"], float(p_trans)

    # Fallback al máximo
    mx = max(p_red, p_trans, p_gr)
    if mx == p_red: return "MADURO", CFG["C_RED"], float(p_red)
    if mx == p_gr: return "INMADURO", CFG["C_GREEN"], float(p_gr)
    return "TRANSICION", CFG["C_ORANGE"], float(p_trans)

def detect_damage(img_bgr, fruit_mask, cnt):
    met = contour_metrics(cnt)
    x, y, w, h = met["x"], met["y"], met["w"], met["h"]

    roi = img_bgr[y:y+h, x:x+w]
    roi_mask = fruit_mask[y:y+h, x:x+w]
    if roi.size == 0: return "OK", 0.0

    # MÁGIA: Ignorar banda superior (donde está el tallo)
    ignore_top = int(h * CFG["DEFECT_IGNORE_TOP_FRAC"])
    if ignore_top > 0:
        roi_mask[:ignore_top, :] = 0

    inside = (roi_mask > 0)
    total = int(np.count_nonzero(inside))
    if total < 200: return "OK", 0.0

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    V = hsv[:, :, 2]

    # Buscamos píxeles anormalmente oscuros
    v_vals = V[inside]
    q = np.quantile(v_vals, CFG["DEFECT_DARK_Q"])
    dark = inside & (V <= int(q))

    ratio = float(np.count_nonzero(dark)) / float(total)

    if ratio >= CFG["DEFECT_RATIO_MIN"]:
        return "DAÑADO", ratio
    return "OK", ratio

## 8. Visualización (UI)

Funciones para dibujar recuadros, textos y el panel informativo semitransparente.

In [ ]:
def draw_panel(img, total, ok_count, maduros, trans, inmaduros, is_tomato, estado_global):
    panel_h = CFG["PANEL_H"]
    overlay = img.copy()
    # Fondo oscuro semitransparente
    cv2.rectangle(overlay, (0, 0), (img.shape[1], panel_h), (30, 30, 30), -1)
    img[:] = cv2.addWeighted(overlay, CFG["ALPHA_PANEL"], img, 1 - CFG["ALPHA_PANEL"], 0)

    # Textos informativos
    put_label(img, "ANALIZADOR DE CALIDAD (JITOMATE) - MODO FOTO", 12, 10, CFG["C_WHITE"], (30,30,30), 0.75, 2)
    put_label(img, f"TOTAL: {total}  (OK: {ok_count})", 12, 48, CFG["C_WHITE"], (30,30,30), 0.70, 2)
    
    txt_j = f"JITOMATE: {'SI' if is_tomato else 'NO'}"
    col_j = CFG["C_GREEN"] if is_tomato else CFG["C_YELLOW"]
    put_label(img, txt_j, 520, 48, col_j, (30,30,30), 0.70, 2)

    put_label(img, f"MADUROS: {maduros}", 12, 78, CFG["C_RED"], (30,30,30), 0.70, 2)
    put_label(img, f"TRANSICION: {trans}", 210, 78, CFG["C_ORANGE"], (30,30,30), 0.70, 2)
    put_label(img, f"INMADUROS: {inmaduros}", 440, 78, CFG["C_GREEN"], (30,30,30), 0.70, 2)

def draw_detection(img, cnt, label, color, idx, quality, score_total, shape_score, calyx_ratio, rip_score, defect_ratio):
    met = contour_metrics(cnt)
    x, y, w, h = met["x"], met["y"], met["w"], met["h"]

    cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
    cv2.drawContours(img, [cnt], -1, color, 2)
    
    # Etiqueta simple arriba de la caja
    # put_label(img, f"{label} | {quality}", x, y - 10, color, (30,30,30))

## 9. Pipeline de Análisis

Esta función coordina todo: recibe imagen -> segmenta -> analiza cada objeto -> dibuja resultados.

In [ ]:
def analyze_image(img_bgr):
    img = resize_for_view(img_bgr)
    out = img.copy()

    mask = build_object_mask(img)
    cnts = extract_instances(mask)

    if len(cnts) == 0:
        draw_panel(out, 0, 0, 0, 0, 0, False, "NO DETECTADO")
        return out, {"total": 0}

    # 1. ¿Es un Jitomate en general? (Basado en el objeto más grande)
    main_cnt = cnts[0]
    main_met = contour_metrics(main_cnt)
    main_mask = mask_from_contour(out.shape, main_cnt)

    shape_sc = tomato_shape_score(main_met)
    calyx_r = calyx_green_ratio(out, main_mask, main_cnt)
    total_sc = tomato_score_total(shape_sc, calyx_r)

    is_tomato = total_sc >= CFG["TOMATO_SCORE_MIN"]
    estado_global = "NO ES JITOMATE"
    if is_tomato:
        estado_global, _, _ = classify_ripeness(out, main_mask)

    # 2. Análisis Individual de cada objeto encontrado
    maduros = trans = inmaduros = 0
    ok_count = 0

    for i, c in enumerate(cnts, 1):
        met = contour_metrics(c)
        if met is None: continue

        fruit_mask = mask_from_contour(out.shape, c)
        shape_score = tomato_shape_score(met)
        calyx_ratio = calyx_green_ratio(out, fruit_mask, c)
        score_total = tomato_score_total(shape_score, calyx_ratio)

        tomato_like = score_total >= CFG["TOMATO_SCORE_MIN"]

        if tomato_like:
            rip_label, color, rip_score = classify_ripeness(out, fruit_mask)
            quality, defect_ratio = detect_damage(out, fruit_mask, c)

            if rip_label == "MADURO": maduros += 1
            elif rip_label == "TRANSICION": trans += 1
            elif rip_label == "INMADURO": inmaduros += 1

            if quality == "OK": ok_count += 1

            draw_detection(out, c, rip_label, color, i, quality,
                           score_total, shape_score, calyx_ratio, rip_score, defect_ratio)
        else:
            # Objeto detectado pero no parece jitomate (ej. Manzana, Pera)
            draw_detection(out, c, "POSIBLE NO-JITOMATE", CFG["C_YELLOW"], i, "N/A",
                           score_total, shape_score, calyx_ratio, 0.0, 0.0)

    draw_panel(out, len(cnts), ok_count, maduros, trans, inmaduros, is_tomato, estado_global)

    return out, {"total": len(cnts), "ok": ok_count, "maduros": maduros, "trans": trans, 
                 "inmaduros": inmaduros, "jitomate": is_tomato, "estado": estado_global}

## 10. Bloque Principal de Ejecución

Ejecuta esta celda para iniciar el programa.
1.  Se abrirá una ventana para elegir archivo.
2.  Se mostrará la imagen procesada.
3.  Presiona **cualquier tecla** o la **X** de la ventana para cerrarla y continuar.

In [ ]:
def main():
    print("=" * 70)
    print("ANALIZADOR DE CALIDAD (JITOMATE) - MODO FOTO")
    print("=" * 70)

    # Bucle simple: procesar una imagen y preguntar si quiere otra
    while True:
        print("\nAbriendo selector de archivos...")
        path = seleccionar_imagen()
        
        if not path:
            print("No seleccionaste ninguna imagen. Finalizando.")
            break

        print(f"Procesando: {os.path.basename(path)}...")
        img = cv2.imread(path)
        if img is None:
            print("Error: No se pudo leer la imagen.")
            continue

        out, info = analyze_image(img)

        # Mostrar ventana
        win_name = "Resultado del Analisis"
        cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
        cv2.imshow(win_name, out)
        
        # Ajustar tamaño y posición
        h, w = out.shape[:2]
        cv2.resizeWindow(win_name, w, h)
        cv2.moveWindow(win_name, 50, 50)

        print("Ventana abierta. Presiona una tecla en la ventana para continuar...")
        _ = wait_until_close(win_name)
        cv2.destroyAllWindows()

        # Reporte en texto
        print("\n--- RESUMEN ---")
        print(f"Total: {info['total']} | Calidad OK: {info['ok']}")
        print(f"Clasificación Global: {'ES JITOMATE' if info['jitomate'] else 'NO ES JITOMATE'}")
        print(f"Detalle: Maduros={info['maduros']}, Transición={info['trans']}, Inmaduros={info['inmaduros']}")

        cont = input("\n¿Analizar otra foto? (s/n): ").strip().lower()
        if cont != "s":
            break

    print("Programa finalizado.")

if __name__ == "__main__":
    main()